In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

housing_path = "df_housing_clean.csv"
main_path    = "df_main_clean.csv"

housing = pd.read_csv("df_housing_clean.csv")
main    = pd.read_csv("df_main_clean.csv")

def pick_col(df, candidates, required=True, name="column"):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(f"Could not find {name}. Tried: {candidates}. Available: {list(df.columns)}")
    return None

# lat/lon
H_LAT = pick_col(housing, ["lat", "latitude", "y"], name="housing latitude")
H_LON = pick_col(housing, ["lon", "lng", "longitude", "x"], name="housing longitude")
M_LAT = pick_col(main,    ["lat", "latitude", "y"], name="stop latitude")
M_LON = pick_col(main,    ["lon", "lng", "longitude", "x"], name="stop longitude")

# price + dates
H_PRICE = pick_col(housing, ["sale_price", "price", "amount"], name="sale price")
H_DATE  = pick_col(housing, ["sale_date", "date", "saledate", "sale_datetime"], name="housing sale date")
M_DATE  = pick_col(main,    ["stop_date", "date", "datetime", "stop_datetime"], name="stop date")

housing[H_DATE] = pd.to_datetime(housing[H_DATE], errors="coerce")
main[M_DATE]    = pd.to_datetime(main[M_DATE], errors="coerce")

# basic cleaning
housing = housing.dropna(subset=[H_LAT, H_LON, H_PRICE, H_DATE]).copy()
main    = main.dropna(subset=[M_LAT, M_LON, M_DATE]).copy()

housing[H_PRICE] = pd.to_numeric(housing[H_PRICE], errors="coerce")
housing = housing.dropna(subset=[H_PRICE])
housing = housing[housing[H_PRICE] > 0].copy()

housing["sale_year"] = housing[H_DATE].dt.year
main["stop_year"]    = main[M_DATE].dt.year

def to_radians(df, lat_col, lon_col):
    coords = df[[lat_col, lon_col]].astype(float).to_numpy()
    return np.radians(coords)

EARTH_RADIUS_MILES = 3958.7613

def wealth_radius(tree, prices, stop_coords_rad, radius_miles):
    r_rad = radius_miles / EARTH_RADIUS_MILES
    neigh = tree.query_radius(stop_coords_rad, r=r_rad, return_distance=False)
    out = np.full(stop_coords_rad.shape[0], np.nan, dtype=float)
    for i, idx in enumerate(neigh):
        if idx.size > 0:
            med = np.median(prices[idx])
            out[i] = np.log(med) if med > 0 else np.nan
    return out

def wealth_knn(tree, prices, stop_coords_rad, k):
    k_eff = min(k, prices.shape[0])
    _, ind = tree.query(stop_coords_rad, k=k_eff)
    med = np.median(prices[ind], axis=1)
    return np.where(med > 0, np.log(med), np.nan)

# ---- Build WealthIndex per stop-year, using only sales <= that year ----
years = sorted(main["stop_year"].dropna().unique())

main["WealthIndex_radius.5mi_past"] = np.nan
main["WealthIndex_radius1mi_past"] = np.nan
main["WealthIndex_knn30_past"] = np.nan

for y in years:
    # restrict sales to <= stop year y
    h_y = housing[housing["sale_year"] <= y]
    if len(h_y) < 10:
        # too few sales to be meaningful; skip
        continue

    tree = BallTree(to_radians(h_y, H_LAT, H_LON), metric="haversine")
    prices = h_y[H_PRICE].to_numpy()

    m_mask = main["stop_year"] == y
    stops_y = main.loc[m_mask, [M_LAT, M_LON]]
    stop_coords = np.radians(stops_y.astype(float).to_numpy())
    
    main.loc[m_mask, "WealthIndex_radius.5mi_past"] = wealth_radius(tree, prices, stop_coords, radius_miles=0.5)
    main.loc[m_mask, "WealthIndex_radius1mi_past"] = wealth_radius(tree, prices, stop_coords, radius_miles=1.0)
    main.loc[m_mask, "WealthIndex_knn30_past"] = wealth_knn(tree, prices, stop_coords, k=30)

# Choose final WealthIndex definition
main["WealthIndex"] = main["WealthIndex_radius1mi_past"]

# Optional: fallback to kNN when radius has no nearby sales
#main["WealthIndex"] = main["WealthIndex"].fillna(main["WealthIndex_knn30_past"])

out_path = "df_main_clean_with_wealth_past_only.csv"
main.to_csv(out_path, index=False)
print("Wrote:", out_path)
print(main["WealthIndex"].describe())

Wrote: df_main_clean_with_wealth_past_only.csv
count    1.344425e+06
mean     1.191835e+01
std      5.328737e-01
min      8.517193e+00
25%      1.164132e+01
50%      1.183320e+01
75%      1.225605e+01
max      1.498978e+01
Name: WealthIndex, dtype: float64
